In [1]:
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.colors as mcl

from files import data

In [5]:
# Make sure this matches project root every time

ROOT = 'SSU-Data-Dash-2026-Toronto-Home-Price-Prediction-Challenge'

if os.path.basename(os.getcwd()) != ROOT:
    os.chdir(ROOT)

os.environ["DATA_DIR"] = "datasets"

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'SSU-Data-Dash-2026-Toronto-Home-Price-Prediction-Challenge'

In [ ]:
listings = data.load("listings")

off_market = data.load("off_market")
list_aware = data.load("list_aware")

In [ ]:
', '.join(listings.columns)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (10, 5))

axes[0].hist(listings['sold_price'], bins = 100)
axes[0].set(title = 'sold_price')

axes[1].hist(np.log(listings['sold_price']), bins = 100, color = 'orange')
axes[1].set(title = 'log(sold_price)')

plt.suptitle('Distribution of sold_price for all listings (in $M)')
plt.show()

In [ ]:
plt.scatter('list_price', 'sold_price', data = listings, marker = '+', color = 'magenta')

plt.title('final sold_price vs. original list_price (in $M)')

plt.xlabel('list_price')
plt.ylabel('sold_price')

plt.show()

In [ ]:
avm_est_available = listings[listings['external_avm_estimate_date'] < listings['date_sold']]

print(f'{len(avm_est_available)} records with external_avm_estimate available before date_sold')
print(f'({np.round(len(avm_est_available) / len(listings) * 100, 2)}% of complete dataset)')

In [ ]:
plt.scatter('external_avm_estimate', 'sold_price', data = avm_est_available, marker = '+', color = 'lightgreen')

plt.title(f'sold_price vs. external_avm_estimate for {len(avm_est_available)} records (in $M)')

plt.xlabel('external_avm_estimate')
plt.ylabel('sold_price')

plt.show()

In [ ]:
listings[['house_type_name', 'sold_price']].groupby('house_type_name').describe().round(2)

In [ ]:
listings[['municipality', 'sold_price']].groupby('municipality').describe().round(2)

In [ ]:
np.random.seed(91)

muni_counts = listings['municipality'].value_counts()

muni_colors = [mcl.to_hex(color) for color in np.random.rand(len(muni_counts), 3)]

muni_counts.sort_index().plot(kind = 'bar', rot = 0, color = muni_colors)

plt.title('Listing counts by municipality')
plt.xlabel(None)

plt.show()

print(muni_counts)

In [ ]:
for i, muni in enumerate(muni_counts.index):
    plt.scatter('latitude', 'longitude', data = listings[listings['municipality'] == muni],
                marker = '^', color = muni_colors[i], alpha = 0.025)

plt.title(f'Approximate map of listings in municipalities')

plt.xlabel('latitude')
plt.ylabel('longitude')

plt.show()

In [ ]:
plt.scatter('latitude', 'longitude', data = listings, marker = '^', color = 'green', alpha = 0.025)

plt.scatter('latitude', 'longitude', data = listings[listings['sold_price'] > listings['sold_price'].median()],
            marker = '^', color = 'magenta', alpha = 0.0125)

plt.title(f'Approximate map of listings')

plt.xlabel('latitude')
plt.ylabel('longitude')

plt.show()

In [ ]:
monthly_means = listings[['date_sold', 'sold_price']].groupby(listings['date_sold'].dt.to_period('M'))['sold_price'].mean()
monthly_medis = listings[['date_sold', 'sold_price']].groupby(listings['date_sold'].dt.to_period('M'))['sold_price'].median()

monthly_means.plot(figsize = (10, 5), kind = 'line', label = 'mean')
monthly_medis.plot(figsize = (10, 5), kind = 'line', label = 'median', color = 'red')

plt.title("sold_price progression from 2021 - 2026 (in $M)")

plt.legend()

plt.show()

In [ ]:
monthly_ht_means = listings[['date_sold', 'house_type_name', 'sold_price']].groupby(
    [listings['date_sold'].dt.to_period('M'), 'house_type_name'])['sold_price'].mean()

monthly_ht_means.unstack().plot(figsize = (10, 5))

plt.title("sold_price means from 2021 - 2026 by house_type (in $M)")

plt.show()

In [ ]:
monthly_ht_medis = listings[['date_sold', 'house_type_name', 'sold_price']].groupby(
    [listings['date_sold'].dt.to_period('M'), 'house_type_name'])['sold_price'].median()

monthly_ht_medis.unstack().plot(figsize = (10, 5))

plt.title("sold_price medians from 2021 - 2026 by house_type (in $M)")

plt.show()

In [ ]:
(listings['date_sold'] - listings['date_listed']).dt.days == listings['days_on_market']

In [ ]:
listings['remarks'].apply(lambda row: len(row.split()))

In [ ]:
def find_word_in_remarks(df, word):
    return df[df['remarks'].apply(lambda row: word.lower() in row.lower())]

In [ ]:
find_word_in_remarks(listings, 'spacious').head(3)

In [ ]:
listings['list_price'].cov(listings['sold_price']) / (listings['list_price'].std() * listings['sold_price'].std())

In [ ]:
homes_sort = listings.sort_values(by = 'date_sold', ascending = True).reset_index(drop = True)

In [ ]:
train_prop = 0.8

n = int(len(homes_sort) * 0.8)
v = len(homes_sort) - n

In [ ]:
X_train = np.concat([np.ones((n, 1)), np.log(np.array(homes_sort[:n]['list_price']).reshape((n, 1)))], axis = 1)

In [ ]:
y_train = np.log(np.array(homes_sort[:n]['sold_price']).reshape((n, 1)))

In [ ]:
B = np.linalg.inv(X_train.T @ X_train) @ X_train.T @ y_train

In [ ]:
X_valid = np.concat([np.ones((v, 1)), np.log(np.array(homes_sort[n:]['list_price']).reshape((v, 1)))], axis = 1)

In [ ]:
y_valid = np.log(np.array(homes_sort[n:]['sold_price']).reshape((v, 1)))

In [ ]:
((y_valid - X_valid @ B) ** 2).sum() / v

In [ ]:
pd.notna(listings['lot_front'] * listings['lot_depth']).sum()

In [ ]:
((pd.notna(listings['lot_front'])) & (pd.notna(listings['lot_depth']))).sum()

In [ ]:
listings[pd.notna(listings).all(axis = 1)]

In [ ]:
from files.features import square_feet

listings['sold_price'] / square_feet(listings['house_area'], adj = 'max')